# 🐦 Ornith-1.5-35B-A3B on Colab — llama.cpp (CUDA) + secured public API

Run the cells top to bottom. At the end you get a **public HTTPS URL, an API key and a ready-to-paste OpenCode config**.

| | |
|---|---|
| Model | `Ornith-1.5-35B-A3B` **IQ4_XS** (~18.7 GB) — Qwen3.5-MoE: 40 layers, 256 experts (8 active), ~3B active params |
| Engine | **llama.cpp `llama-server`**, prebuilt CUDA binary (no compile), OpenAI-compatible `/v1` |
| Runtime | **T4 GPU** (free tier) |
| Context | **128K** by default (native 262K, set in cell 1) |
| Target | **25+ tok/s** (Ollama gave ~12–14 on the same T4) |

### How it fits and why it is faster than Ollama
The file (18.7 GB) is bigger than the T4's 15 GB VRAM. llama.cpp lets us choose **exactly what goes where**:
- **GPU:** attention, shared experts, router, embeddings, the KV cache, and the routed experts of as many layers as fit.
- **CPU/RAM:** only the routed experts of the remaining layers (`--n-cpu-moe`). Only 8 of 256 experts run per token, so the CPU part is light.
- The notebook **computes the split from your free VRAM *and* RAM** and retries automatically if it guesses too high.
- **RAM matters as much as VRAM on Colab (~12.7 GB).** If the CPU-side experts don't fit in RAM, Linux keeps re-reading them from disk (we measured ~150 MB of disk reads per token = 2-3 tok/s). So the CPU part is **loaded into RAM (`--load-mode none`, formerly `--no-mmap`)** and kept small by using the 18.7 GB IQ4_XS file instead of the 21.7 GB Q4_K_M.
- Long context is cheap here: only 10 of 40 layers use full attention (the other 30 are linear), so the KV cache is small.

> ⚠️ Keep the browser tab open (free Colab disconnects on idle). The tunnel URL changes every run.

In [ ]:
# 1) SETTINGS + hardware check  (edit the CFG values, then run)
import subprocess, shutil, json, os

CFG = {
    # --- model ---------------------------------------------------------------
    'QUANT': 'IQ4_XS',      # 'IQ4_XS' (18.7 GB, default: fits Colab RAM) | 'Q4_K_M' (21.7 GB: its CPU part does NOT fit Colab's RAM -> slow disk thrashing)
    # --- context -------------------------------------------------------------
    'CTX': 131072,          # 131072 = 128K (default). 262144 = full native 256K (costs ~1.4 GB more VRAM -> a few tok/s less)
    'KV': 'q8_0',           # KV-cache precision: 'q8_0' (recommended) | 'f16' (double the memory) | 'q4_0' (half)
    # --- speed ---------------------------------------------------------------
    'N_CPU_MOE': 'auto',    # 'auto' = computed from free VRAM. Or a number (layers whose experts stay on CPU). LOWER = faster, until VRAM is full
    'UB': 1024,             # prompt micro-batch. Bigger = much faster prompt processing, needs a bit more VRAM
    'THREADS': os.cpu_count() or 2,
    'MMAP': 'auto',         # 'auto' = load CPU experts into RAM (no mmap) when they fit -> no disk thrashing | 'off' = always load into RAM | 'on' = memory-map from disk
    # --- behaviour -----------------------------------------------------------
    'REASONING': 'on',      # 'on' = model thinks first (default Ornith behaviour) | 'off' = answers directly (faster, shorter)
    'API_KEY': 'ollama',    # what OpenCode sends. Use secrets.token_urlsafe(16) for a private random key
    'PORT': 11434,
    'ALIAS': 'ornith-1.5-35b-a3b',   # model id served by the API
    'EXTRA': [],            # any extra llama-server flags, e.g. ['--no-webui']
}

# --- hardware report ---------------------------------------------------------
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
gpu = sh('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader')
assert gpu, 'No GPU found: Runtime > Change runtime type > T4 GPU'
mem = {l.split(':')[0]: int(l.split()[1]) for l in open('/proc/meminfo') if l.split()[0].rstrip(':') in ('MemTotal', 'MemAvailable')}
ram_total = mem['MemTotal'] / 1048576
ram_avail = mem['MemAvailable'] / 1048576
free_disk = shutil.disk_usage('/').free / 1e9
print('GPU      :', gpu)
print(f'CPU      : {os.cpu_count()} vCPU')
print(f'RAM      : {ram_total:.1f} GB total, {ram_avail:.1f} GB available')
print(f'Disk     : {free_disk:.0f} GB free (need ~25 GB)')
if ram_total < 11:
    print('WARNING: under 11 GB RAM. The CPU-side experts may not fit in RAM; use QUANT = "IQ4_XS" and a smaller CTX.')
if free_disk < 25:
    print('WARNING: under 25 GB free disk.')

json.dump(CFG, open('/content/config.json', 'w'))
print('\nsettings saved:', {k: v for k, v in CFG.items() if k != 'EXTRA'})

In [ ]:
# 2) Install llama.cpp (prebuilt CUDA binary) + CUDA libs + the Colab driver-library fix. No compile.
import subprocess, re, glob, os, site, json, urllib.request

CFG = json.load(open('/content/config.json'))
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True)

def gh_json(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'colab-llama'})
    return json.load(urllib.request.urlopen(req, timeout=30))

# Candidate builds, best first: newest official CUDA build -> pinned official build -> PrismML fork (proven on this Colab)
cands = []
try:
    rel = gh_json('https://api.github.com/repos/ggml-org/llama.cpp/releases/latest')
    tag = rel['tag_name']
    have = {a['name']: a['browser_download_url'] for a in rel['assets']}
    for v in ('12.8', '12.4'):
        n, c = f'llama-{tag}-bin-ubuntu-cuda-{v}-x64.tar.gz', f'cudart-llama-{tag}-bin-ubuntu-cuda-{v}-x64.tar.gz'
        if n in have:
            cands.append((f'ggml-org {tag} CUDA {v}', have[n], have.get(c)))
except Exception as e:
    print('GitHub API unavailable, using pinned builds:', e)
S = 'https://github.com/ggml-org/llama.cpp/releases/download/b12359/'
cands.append(('ggml-org b12359 CUDA 12.8 (pinned)', S + 'llama-b12359-bin-ubuntu-cuda-12.8-x64.tar.gz', S + 'cudart-llama-b12359-bin-ubuntu-cuda-12.8-x64.tar.gz'))
F = 'https://github.com/PrismML-Eng/llama.cpp/releases/download/prism-b12128-d7b2896/'
cands.append(('PrismML fork b12128 CUDA 12.4 (fallback)', F + 'llama-prism-b12128-d7b2896-bin-linux-cuda-12.4-x64.tar.gz', None))

def build_env(use_pip):
    """LD_LIBRARY_PATH with: binary libs, CUDA runtime libs, and THE KEY FIX: the NVIDIA driver dir (libcuda.so.1)."""
    drv = glob.glob('/usr/lib64-nvidia/libcuda.so*') + glob.glob('/usr/lib/x86_64-linux-gnu/libcuda.so*') + glob.glob('/usr/local/nvidia/lib64/libcuda.so*')
    assert drv, 'libcuda.so.1 not found: runtime has no working GPU (Runtime > Change runtime type > T4)'
    dirs = [os.path.dirname(x) for x in glob.glob('/content/pkg/**/lib*.so*', recursive=True)]
    dirs += [os.path.dirname(x) for x in glob.glob('/content/stock/**/lib*.so*', recursive=True)]
    if use_pip:
        for sd in site.getsitepackages():
            dirs += [os.path.dirname(x) for x in glob.glob(os.path.join(sd, 'nvidia', '*', 'lib', 'lib*.so*'))]
    dirs += [os.path.dirname(x) for x in drv]
    dirs += [p for p in os.environ.get('LD_LIBRARY_PATH', '').split(':') if p]
    return 'LD_LIBRARY_PATH=' + ':'.join(dict.fromkeys(dirs))

def try_install(label, bin_url, cudart_url):
    print(f'\n>> trying: {label}')
    sh('rm -rf /content/pkg /content/stock /content/bin.tgz /content/cudart.tgz; mkdir -p /content/pkg /content/stock')
    if sh(f'curl -fsSL -o /content/bin.tgz "{bin_url}"').returncode != 0:
        print('   download failed'); return None
    sh('tar -xzf /content/bin.tgz -C /content/pkg')
    found = glob.glob('/content/pkg/**/llama-server', recursive=True)
    if not found:
        print('   llama-server not in archive'); return None
    exe = found[0]; os.chmod(exe, 0o755)
    if cudart_url and sh(f'curl -fsSL -o /content/cudart.tgz "{cudart_url}"').returncode == 0:
        sh('tar -xzf /content/cudart.tgz -C /content/stock')
    for use_pip in (False, True):
        if use_pip:
            print('   no GPU backend yet, adding pip CUDA 12 libs...')
            sh('pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12')
        env = build_env(use_pip)
        dev = sh(f'{env} "{exe}" --list-devices 2>&1 | head -12').stdout
        if re.search(r'CUDA\d*:', dev):
            return exe, env, dev
    print('   CUDA backend did not load:\n', dev)
    return None

result = None
for c in cands:
    result = try_install(*c)
    if result:
        print('   OK:', c[0]); break
assert result, 'No llama.cpp build could use the GPU. Paste the output above for diagnosis.'
exe, env, dev = result
open('/content/llama_bin.txt', 'w').write(exe)
open('/content/llama_env.txt', 'w').write(env)
print('\nllama-server:', exe)
print(dev)
print(sh(f'{env} "{exe}" --version 2>&1 | head -3').stdout)

In [ ]:
# 3) Download the model (IQ4_XS ~18.7 GB, Q4_K_M ~21.7 GB). Resumes if interrupted: just re-run.
import subprocess, os, json, glob, urllib.request

CFG = json.load(open('/content/config.json'))
FILES = {   # quant -> (repo, preferred filename, minimum plausible size in GB)
    'Q4_K_M': ('ornith-ai/Ornith-1.5-35B-A3B-GGUF', 'Ornith-1.5-35B-Q4_K_M.gguf', 19.0),
    'IQ4_XS': ('AtomicChat/Ornith-1.5-35B-A3B-GGUF', 'Ornith-1.5-35B-A3B-IQ4_XS.gguf', 17.0),
}
repo, fname, min_gb = FILES[CFG['QUANT']]

try:    # confirm the file exists in the repo; otherwise pick the closest .gguf by name
    info = json.load(urllib.request.urlopen(urllib.request.Request(
        f'https://huggingface.co/api/models/{repo}', headers={'User-Agent': 'colab-llama'}), timeout=30))
    files = [x['rfilename'] for x in info.get('siblings', []) if x['rfilename'].endswith('.gguf')]
    if files and fname not in files:
        alt = [f for f in files if CFG['QUANT'].lower() in f.lower() and 'mmproj' not in f.lower() and '-of-' not in f]
        print('Available GGUF files:', files)
        assert alt, f'{fname} not found in {repo}: pick another QUANT in cell 1'
        fname = alt[0]; print('using', fname)
except AssertionError:
    raise
except Exception as e:
    print('HF API check skipped:', e)

path = f'/content/{fname}'
for old in glob.glob('/content/Ornith*.gguf'):      # free disk + page cache from a previously downloaded other quant
    if old != path and os.path.getsize(old) > 1e9:
        print('removing unused model file:', old); os.remove(old)
subprocess.run(f'wget -c -q --show-progress -O "{path}" "https://huggingface.co/{repo}/resolve/main/{fname}"', shell=True, check=True)
size = os.path.getsize(path) / 1e9
print(f'{fname}: {size:.2f} GB')
assert size > min_gb, 'Download incomplete, re-run this cell to resume'
CFG['MODEL'], CFG['MODEL_BYTES'] = path, os.path.getsize(path)
json.dump(CFG, open('/content/config.json', 'w'))

In [ ]:
# 4) Server helpers + START llama-server  (auto-sizes the GPU/CPU expert split, retries on out-of-memory)
import subprocess, time, json, urllib.request, urllib.error, os, re, shlex

CFG = json.load(open('/content/config.json'))
LLAMA = open('/content/llama_bin.txt').read().strip()
ENV = open('/content/llama_env.txt').read().strip()
BASE = f"http://127.0.0.1:{CFG['PORT']}"
LAYERS = 40

# Ask the binary which flags it knows: llama.cpp renames/removes flags between builds (e.g. --no-mmap -> --load-mode none)
HELP = subprocess.run(f'{ENV} "{LLAMA}" --help 2>&1', shell=True, capture_output=True, text=True).stdout
NOMMAP_ARGS = ['--load-mode', 'none'] if '--load-mode' in HELP else ['--no-mmap']
for _f in ('--n-cpu-moe', '--chat-template-kwargs', '--jinja', '--api-key'):
    if _f not in HELP:
        print(f'WARNING: this llama.cpp build does not list {_f}; the server may refuse to start.')
print('load-without-mmap flag for this build:', ' '.join(NOMMAP_ARGS))
os.makedirs('/content/logs', exist_ok=True)
PROC = None

def save_cfg():
    json.dump(CFG, open('/content/config.json', 'w'))

def log_tail(n=2500):
    return open('/content/logs/llama-server.log', errors='ignore').read()[-n:]

def stop_server():
    global PROC
    if PROC and PROC.poll() is None:
        PROC.terminate()
        try: PROC.wait(timeout=15)
        except Exception: PROC.kill()
    subprocess.run(['pkill', '-x', 'llama-server'])     # -x = exact process name
    time.sleep(2)

def vram_total_gib():
    o = subprocess.run('nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits', shell=True, capture_output=True, text=True).stdout
    return int(o.split()[0]) / 1024

def plan_cpu_moe(kv):
    """How many layers keep their routed experts on the CPU so everything else fits in VRAM."""
    model_gib = CFG['MODEL_BYTES'] / 2**30
    nonexpert = 2.3                                   # attention/linear-attn, shared experts, router, embeddings, output head
    per_layer = (model_gib - nonexpert) / LAYERS      # routed experts per layer
    kv_bytes_per_tok = {'f16': 20480, 'q8_0': 10880, 'q4_0': 5760}[kv]   # only 10 of 40 layers use full attention
    kv_gib = CFG['CTX'] * kv_bytes_per_tok / 2**30
    compute = 0.6 + CFG['UB'] / 1024 * 0.6            # CUDA compute buffers grow with the micro-batch
    budget = vram_total_gib() - 0.7 - compute - kv_gib - 0.5   # driver context + safety margin
    on_gpu = max(0, int((budget - nonexpert) / per_layer))
    n = min(LAYERS, max(0, LAYERS - on_gpu))
    return n, dict(per_layer=per_layer, kv=kv_gib, compute=compute, budget=budget, cpu_gib=n * per_layer)

def ram_total_gib():
    return int([l for l in open('/proc/meminfo') if l.startswith('MemTotal')][0].split()[1]) / 2**20

def ram_fits(cpu_gib):
    """Can the CPU-side experts live in RAM? Reserve ~4 GB for Python, the server's own buffers and the OS, plus 1 GB margin."""
    return cpu_gib <= ram_total_gib() - 4.0 - 1.0

def sampling_args():
    if CFG.get('REASONING', 'on') == 'off':   # non-thinking mode
        return ['--temp', '0.7', '--top-p', '0.8', '--top-k', '20', '--min-p', '0.0', '--presence-penalty', '1.5',
                '--chat-template-kwargs', json.dumps({'enable_thinking': False})]
    return ['--temp', '0.6', '--top-p', '0.95', '--top-k', '20', '--min-p', '0.0']   # Ornith's recommended general settings

def _launch(n_moe, kv, extra, wait, nommap):
    global PROC
    stop_server()
    args = [
        '-m', CFG['MODEL'], '--alias', CFG['ALIAS'],
        '-ngl', '99', '--n-cpu-moe', str(n_moe),      # all layers on GPU, but experts of the first n_moe layers stay on CPU
        '-fa', 'on', '-c', str(CFG['CTX']), '-ctk', kv, '-ctv', kv,
        '-np', '1',                                    # one slot = the whole context for one session
        '-b', '2048', '-ub', str(CFG['UB']),
        '-t', str(CFG['THREADS']),
        '--jinja',                                     # chat template + native tool calling (needed by OpenCode)
        '--host', '127.0.0.1', '--port', str(CFG['PORT']),
    ] + sampling_args()
    if CFG.get('API_KEY'):
        args += ['--api-key', CFG['API_KEY']]
    if nommap:
        args += NOMMAP_ARGS                            # CPU experts are read into RAM once and stay there (no disk thrashing)
    args += list(extra)
    cmd = f'exec env {shlex.quote(ENV)} {shlex.join([LLAMA] + args)} > /content/logs/llama-server.log 2>&1'
    PROC = subprocess.Popen(['bash', '-c', cmd])
    t0 = time.time()
    while time.time() - t0 < wait:
        time.sleep(3)
        if PROC.poll() is not None:                    # real exit detection: fails in seconds
            raise RuntimeError(f'llama-server exited (code {PROC.returncode}):\n' + log_tail())
        try:
            if urllib.request.urlopen(BASE + '/health', timeout=5).status == 200:
                CFG['CTX_USED'], CFG['N_CPU_MOE_USED'], CFG['KV_USED'], CFG['NOMMAP_USED'] = CFG['CTX'], n_moe, kv, nommap
                save_cfg()
                print(f'server ready in {time.time()-t0:.0f}s | ctx {CFG["CTX"]:,} | KV {kv} | experts on CPU: {n_moe}/{LAYERS} layers '
                      f'| weights in RAM: {"yes (" + " ".join(NOMMAP_ARGS) + ")" if nommap else "no (memory-mapped)"} | reasoning {CFG["REASONING"]}')
                return
        except Exception:
            pass
    raise RuntimeError('llama-server did not become healthy in time:\n' + log_tail())

OOM = re.compile(r'out of memory|cudaMalloc|failed to allocate|unable to allocate|alloc.*failed', re.I)
HOSTERR = re.compile(r'cudaHostAlloc|pinned|CUDA_Host|CPU buffer|CPU_Mapped|Killed|code -9', re.I)   # host-RAM problems, not VRAM

def start_server(extra=(), wait=900):
    kv = CFG.get('KV', 'q8_0')
    if CFG.get('N_CPU_MOE', 'auto') == 'auto':
        n, info = plan_cpu_moe(kv)
        print(f'auto split: {LAYERS-n}/{LAYERS} layers fully on GPU, {n} layers with experts on CPU '
              f'(VRAM budget {info["budget"]:.1f} GiB, KV {info["kv"]:.2f} GiB, {info["per_layer"]:.2f} GiB/layer)')
    else:
        n = int(CFG['N_CPU_MOE'])
    cpu_gib = n * (CFG['MODEL_BYTES'] / 2**30 - 2.3) / LAYERS
    mm = CFG.get('MMAP', 'auto')
    nommap = (mm == 'off') or (mm == 'auto' and ram_fits(cpu_gib))
    print(f'CPU-side experts: {cpu_gib:.1f} GiB | RAM {ram_total_gib():.1f} GiB -> ' +
          (f'loading them into RAM ({" ".join(NOMMAP_ARGS)})' if nommap else 'memory-mapped from disk'))
    if not nommap and mm == 'auto':
        print(f'WARNING: {cpu_gib:.1f} GiB of CPU experts will NOT fit in {ram_total_gib():.1f} GiB RAM: expect disk thrashing (2-3 tok/s).\n'
              '         Fix: QUANT = "IQ4_XS" and/or a smaller CTX in cell 1, then re-run cells 1, 3, 4, 5.')
    for attempt in range(7):
        try:
            return _launch(n, kv, extra, wait, nommap)
        except RuntimeError as e:
            msg = str(e)
            bad = re.search(r'invalid argument: (\S+)', msg)
            if bad:                                                   # a flag this build does not know: retrying cannot help
                print(f'This llama.cpp build rejects the flag {bad.group(1)}. Check `llama-server --help`, or tell me and I will update the notebook.')
                raise
            if nommap and HOSTERR.search(msg):                       # host RAM could not hold it -> fall back to mmap
                print('not enough host RAM for --no-mmap, falling back to memory-mapped loading...')
                nommap = False; continue
            if OOM.search(msg) and n < LAYERS:                        # GPU memory ran out -> put 3 more layers' experts on CPU
                n = min(LAYERS, n + 3)
                print(f'out of VRAM: moving 3 more layers of experts to CPU (now {n}) and retrying...')
                continue
            print(msg); raise
    raise RuntimeError('could not fit the model; lower CTX or UB in cell 1')

def api(path, body, timeout=1200, key=None):
    k = CFG.get('API_KEY') if key is None else key
    headers = {'Content-Type': 'application/json'}
    if k: headers['Authorization'] = 'Bearer ' + k
    req = urllib.request.Request(BASE + path, data=json.dumps(body).encode(), headers=headers)
    return json.loads(urllib.request.urlopen(req, timeout=timeout).read())

start_server(CFG['EXTRA'])

In [ ]:
# 5) Smoke test + resource report + speed benchmark (target: 25 tok/s)
import subprocess, json

def gpu_mem():
    o = subprocess.run('nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader,nounits', shell=True, capture_output=True, text=True).stdout.split(',')
    return int(o[0]), int(o[1])

used, total = gpu_mem()
print(f'VRAM: {used} / {total} MiB used')
assert used > 4000, 'Model is NOT on the GPU (silent CPU fallback). Log:\n' + log_tail(1500)

out = api('/v1/chat/completions', {'model': CFG['ALIAS'], 'max_tokens': 1500,
          'messages': [{'role': 'user', 'content': 'Reply with exactly: MODEL LOADED'}]})
print('reply:', out['choices'][0]['message']['content'].strip()[:100])

PROMPT = ('# Python 3\n# LRU cache with get/put in O(1) plus unit tests\n'
          'from collections import OrderedDict\n\nclass LRUCache:\n')
LONG = 'def add(a, b):\n    return a + b\n\n' * 330      # roughly 3,000 tokens

def disk_read_mb():                      # data read from disk since boot (KiB in /proc/vmstat)
    return int([l for l in open('/proc/vmstat') if l.startswith('pgpgin ')][0].split()[1]) / 1024

def bench(n=200, runs=3):
    api('/completion', {'prompt': PROMPT, 'n_predict': 16, 'cache_prompt': False})     # warm-up
    sp = []
    for _ in range(runs):
        r = api('/completion', {'prompt': PROMPT, 'n_predict': n, 'temperature': 0.2, 'cache_prompt': False, 'ignore_eos': True})
        sp.append(r['timings']['predicted_per_second'])
    return sum(sp) / len(sp), sp

d0 = disk_read_mb()
tg, runs = bench()
disk_per_tok = (disk_read_mb() - d0) / (200 * 3)
pp = api('/completion', {'prompt': LONG, 'n_predict': 1, 'cache_prompt': False})['timings']
print(f'\nDECODE        : {tg:.1f} tok/s   (runs: {", ".join(f"{s:.1f}" for s in runs)})')
print(f'PROMPT INPUT  : {pp["prompt_per_second"]:.0f} tok/s  ({pp["prompt_n"]} tokens)')

used, total = gpu_mem()
free_gib = (total - used) / 1024
mem = {l.split(':')[0]: int(l.split()[1]) for l in open('/proc/meminfo') if l.split()[0].rstrip(':') in ('MemTotal', 'MemAvailable')}
print(f'VRAM free     : {free_gib:.1f} GiB | RAM available: {mem["MemAvailable"]/1048576:.1f} GB')
print(f'DISK READS    : {disk_per_tok:.1f} MB per generated token  (should be ~0)')

n_used = CFG['N_CPU_MOE_USED']
per_layer = (CFG['MODEL_BYTES'] / 2**30 - 2.3) / LAYERS
spare_layers = int(max(0, free_gib - 0.8) / per_layer)
if disk_per_tok > 5:
    print(f'\n🛑 DISK THRASHING: {disk_per_tok:.0f} MB read from disk per token. The CPU-side weights do not fit in RAM, so Linux re-reads them constantly.')
    print('  Fix, biggest win first:')
    print('  1. QUANT = "IQ4_XS" (if not already) and CTX = 65536 in cell 1, then re-run cells 1, 3, 4, 5.')
    print('  2. Check cell 4 printed "loading them into RAM (--no-mmap)"; if it printed "memory-mapped", the CPU part is too big for RAM.')
    print('  3. Runtime > Disconnect and delete runtime, then start fresh (a stale session may hold RAM).')
elif tg >= 25:
    print(f'\n✅ Target reached ({tg:.1f} >= 25 tok/s).')
else:
    print(f'\n⚠️ Below 25 tok/s, and no disk thrashing. Options, biggest win first:')
    if spare_layers >= 1:
        print(f'  1. {free_gib:.1f} GiB of VRAM is still free: try retune({max(0, n_used - spare_layers)}) below (puts {spare_layers} more layers on GPU; this also shrinks the RAM part).')
    print('  2. CTX = 65536 in cell 1 (frees VRAM -> more layers on GPU), then re-run cells 1, 4, 5.')
    print('  3. REASONING = "off" in cell 1 for faster answers (no thinking tokens).')

def retune(n_moe):
    """Restart with a specific number of CPU-expert layers and re-measure. Lower n = more on GPU = faster (until VRAM runs out)."""
    CFG['N_CPU_MOE'] = n_moe; save_cfg()
    start_server(CFG['EXTRA'])
    t, r = bench()
    print(f'n_cpu_moe={n_moe}: {t:.1f} tok/s  (runs: {", ".join(f"{s:.1f}" for s in r)})')

In [ ]:
# 6) Public tunnel (Cloudflare quick tunnel, no account) + ready-to-paste OpenCode config
import subprocess, time, re, json

CFG.update(json.load(open('/content/config.json')))
subprocess.run('pkill -f "[c]loudflared"; true', shell=True)
subprocess.run('curl -fsSL -o /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared', shell=True, check=True)
subprocess.Popen(['bash', '-c',
    f"/content/cloudflared tunnel --url http://127.0.0.1:{CFG['PORT']} --no-autoupdate > /content/logs/tunnel.log 2>&1"])
url = None
for _ in range(40):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/logs/tunnel.log', errors='ignore').read())
    if m:
        url = m.group(0); break
if not url:
    print(open('/content/logs/tunnel.log', errors='ignore').read()[-800:])
    raise RuntimeError('tunnel did not come up')
time.sleep(5)  # let the edge register the tunnel
CFG['URL'] = url
save_cfg()

def print_opencode():
    ctx = CFG.get('CTX_USED', CFG['CTX'])
    print('PUBLIC URL :', CFG['URL'])
    print('OpenAI base:', CFG['URL'] + '/v1')
    print('API key    :', CFG['API_KEY'] or '(none; any value works)')
    print('Model id   :', CFG['ALIAS'], '| context:', f'{ctx:,}', '\n')
    oc = {
        '$schema': 'https://opencode.ai/config.json',
        'provider': {'colab-llamacpp': {
            'npm': '@ai-sdk/openai-compatible', 'name': 'Ornith 1.5 35B-A3B (Colab)',
            'options': {'baseURL': CFG['URL'] + '/v1', 'apiKey': CFG['API_KEY'] or 'none'},
            'models': {CFG['ALIAS']: {'name': 'Ornith-1.5-35B-A3B',
                                      'limit': {'context': ctx, 'output': 16384}}}}}}
    print('--- paste into opencode.json (~/.config/opencode/opencode.json) ---')
    print(json.dumps(oc, indent=2))

print_opencode()

In [ ]:
# 7) Verify the public URL end-to-end
import json, urllib.request
CFG.update(json.load(open('/content/config.json')))
h = {'Content-Type': 'application/json', 'User-Agent': 'curl/8.0'}   # Cloudflare blocks the default Python user-agent
if CFG['API_KEY']: h['Authorization'] = 'Bearer ' + CFG['API_KEY']
req = urllib.request.Request(CFG['URL'] + '/v1/chat/completions',
    data=json.dumps({'model': CFG['ALIAS'], 'max_tokens': 1500,
                     'messages': [{'role': 'user', 'content': 'Reply with exactly: API IS LIVE'}]}).encode(), headers=h)
out = json.loads(urllib.request.urlopen(req, timeout=900).read())
print(out['choices'][0]['message']['content'].strip()[:300])
print('\nTimings:', {k: round(v, 1) for k, v in out.get('timings', {}).items() if k.endswith('per_second')})

In [ ]:
# 8) (Optional) FIX / CHANGE THE API KEY without losing the tunnel URL.
#    Use when OpenCode gets "401 Unauthorized": the server only accepts the exact key it was started with.
#    Restarts the server with the FULL config (tool calling, context, speed flags) and the key you choose.
NEW_KEY = 'ollama'    # must equal apiKey in OpenCode. '' = disable authentication (anyone with the URL can use it)

import urllib.request, urllib.error
CFG['API_KEY'] = NEW_KEY
save_cfg()
start_server(CFG['EXTRA'])

def status(key, base=BASE, extra_headers=None):
    h = dict(extra_headers or {})
    if key is not None:
        h['Authorization'] = 'Bearer ' + key
    try:
        return urllib.request.urlopen(urllib.request.Request(base + '/v1/models', headers=h), timeout=30).status
    except urllib.error.HTTPError as e:
        return e.code

print('local, correct key :', status(NEW_KEY), '(expect 200)')
print('local, wrong key   :', status('wrong-key'), '(expect 401)' if NEW_KEY else '(no auth: 200 is expected)')
if CFG.get('URL'):
    print('public, correct key:', status(NEW_KEY, CFG['URL'], {'User-Agent': 'curl/8.0'}), '(expect 200)\n')
    print_opencode()
else:
    print('Tunnel not started yet: run cell 6 next.')

In [ ]:
# 9) (Optional) Keep-alive monitor: keeps the session busy and restarts the server/tunnel if they die.
#    Leave this cell running; stop it with the square button.
import time, subprocess, urllib.request, json
CFG.update(json.load(open('/content/config.json')))
while True:
    time.sleep(60)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{CFG['PORT']}/health", timeout=10)
    except Exception:
        print('server down, restarting...'); start_server(CFG['EXTRA'])
    if subprocess.run('pgrep -f "[c]loudflared"', shell=True, capture_output=True).returncode != 0:
        print('tunnel down: re-run cell 6 (URL will change)'); break

## 🔌 Connect OpenCode (or any OpenAI client)
Cell 6 prints a complete `opencode.json` block with your URL, key and context size. Merge it into your OpenCode config and pick **Ornith 1.5 35B-A3B (Colab)**.
Other clients: `base_url = <URL>/v1`, `api_key = <printed key>` (default `ollama`), model `ornith-1.5-35b-a3b`.

## 🎛️ Tuning guide (all in cell 1)
| Goal | Change |
|---|---|
| **More speed** | Lower `N_CPU_MOE` (cell 5 suggests a value, or run `retune(N)`), or lower `CTX` (frees VRAM for more GPU layers) |
| **Higher quality** | `QUANT = 'Q4_K_M'` — but its CPU part (~9 GiB) does **not** fit Colab's RAM and will thrash the disk (2–3 tok/s). Only with a high-RAM runtime |
| **More context** | `CTX = 262144` (full native). Costs ~1.4 GB VRAM → ~3 fewer GPU layers → a few tok/s less |
| **More VRAM headroom** | `CTX = 65536` or `UB = 512` (slower prompt input) |
| **Faster answers** | `REASONING = 'off'` (no thinking tokens; good for quick edits) |
| **Better prompt input speed** | `UB = 2048` if VRAM allows (re-run from cell 4) |

## Notes
- **New VM / restart:** re-run cells 1→7 (binary ~2 min, model ~5–8 min). Cell 8 only for 401 errors. New tunnel URL each time.
- **First load is slow** (reading ~19 GB from disk, 1–3 min). Later restarts are faster (file cached).
- **Why load into RAM (`--load-mode none`, formerly `--no-mmap`):** memory-mapping lets Linux evict model pages when RAM is tight, and the server then re-reads them from disk on every token. Loading the CPU experts into RAM once avoids that. Cell 5 prints "DISK READS" per token to prove it (should be ~0).
- **Speed depends on the Colab CPU** (2 vCPU): the CPU-side experts set the pace. Expect ~2x the Ollama result; cell 5 prints the real number.
- Decode speed drops a little as the context fills. Prompt input (~thousands of tok/s on GPU layers, slower on CPU layers) matters for big agent prompts.
- The model has a built-in multi-token-prediction head, but speculative decoding is **not** enabled here: with experts on the CPU it adds work instead of saving it.
- Out-of-memory at load? The notebook retries with more experts on CPU automatically; if it still fails, lower `CTX`/`UB`.
- If tok/s is very low, check cell 5: VRAM used should be ~12–14 GB, not 0.